# Homemade NN implementation

### Libary and data imports

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import random

In [29]:
train=pd.read_csv("../data/case1/train1.csv")
test=pd.read_csv("../data/case1/test1.csv")
for df in (train, test):
    df["n"] = (df["speed_rpm"] / 1000)
    df["Q"] = df["flow_m3_h"]
    df["P"] = df["power_w"]
    df["deltaP"] = (df["outlet_pressure_bar"] - df["inlet_pressure_bar"])

features = ["n", "P", "deltaP"]
X, y= train[features], train["Q"]
X_test, y_test= test[features], test["Q"]

X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.8, random_state=42)

In [30]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

### NN Functions

In [31]:
def inputs(X, y, batch_size):
    batch = []
    y_values = []
    nums = random.sample(range(len(X)), batch_size)
    for num in nums:
        feature_values = X[num]
        batch.append(feature_values)
        y_values.append(np.array(y)[num])
    return batch, y_values

def find_weights(neurons, input, output):
    weights = []
    for _ in range(neurons):
        a = np.random.randn(output, input) * np.sqrt(2 / input)
        a = a.tolist()[0]
        weights.append(a)
    return np.array(weights).T

def bias(neurons):
    return np.zeros(neurons)  

def Z(features, weights, bias_):
    return features @ weights + bias_

def relu(Z):
    return np.maximum(0, Z)

def sigmoid(Z):
    return (1/ (1+np.exp(-Z)))

def linear(Z):
    return (Z).sum(axis=1)

def MSE(pred, y):
    return np.mean((pred - y) ** 2)


### Iniziating a 3-3-1 NN

In [39]:
feature_values, y_true = inputs(X_train_scaled, y_train, 5)
feature_values = np.array(feature_values)
y_true = np.array(y_true).reshape(-1, 1)
weights1 = find_weights(3, 3, 1)
bias1= bias(3)
weights2 = find_weights(1, 3, 1)
bias2=bias(1)

In [40]:
Z_1 = Z(feature_values, weights1, bias1)
A_1 = relu(Z_1)
Z_2 = Z(A_1, weights2, bias2)
output = Z_2
error = MSE(output, y_true)

In [41]:
error

np.float64(890.7040518772399)

In [42]:
learning_rate = 0.5
dL_doutput = (2 / len(y_true)) * (output - y_true)

gradient_weight2 = (A_1.T @ dL_doutput).reshape(-1, 1)

new_weight2 = weights2 - learning_rate * gradient_weight2


In [ ]:

gradient_z = (2 / len(y_true)) * (np.sum(output - y_true))
gradient_weights = gradient_z * A_1

In [ ]:
gradient_weight2 = np.average(gradient_weights, axis=0)
new_weight2 = weights2.flatten() - (learning_rate * gradient_weight2)

In [ ]:
relu_grad = (A_1 > 0).astype(int)

In [ ]:
(A_1 > 0).astype(int) * gradient_weight2.T

array([[-16.08488813, -24.93270787,  -0.        ],
       [ -0.        ,  -0.        , -15.21804866],
       [ -0.        , -24.93270787,  -0.        ],
       [-16.08488813, -24.93270787,  -0.        ],
       [-16.08488813, -24.93270787,  -0.        ]])